In [40]:
%pylab inline

Populating the interactive namespace from numpy and matplotlib


In [41]:
sql = '''
SELECT "OFFENSE_DESCRIPTION",
       "OCCURRED_ON_DATE"
FROM "b973d8cb-eeb2-4e7e-99da-c92938efc9c0"
LIMIT 10000
'''

In [42]:
bpd_logs = "https://data.boston.gov/api/3/action/datastore_search_sql"

In [43]:
import urllib, requests

In [44]:
bpd_logs = "https://data.boston.gov/api/3/action/datastore_search_sql"
url = bpd_logs + "?sql=" + urllib.parse.quote(sql)

rows = requests.get(url).json()
records = rows['result']['records']

In [45]:
records

[{'OFFENSE_DESCRIPTION': 'INVESTIGATE PERSON',
  'OCCURRED_ON_DATE': '2023-01-27 22:44:00+00'},
 {'OFFENSE_DESCRIPTION': 'VERBAL DISPUTE',
  'OCCURRED_ON_DATE': '2023-01-17 20:21:00+00'},
 {'OFFENSE_DESCRIPTION': 'INVESTIGATE PERSON',
  'OCCURRED_ON_DATE': '2023-01-24 00:00:00+00'},
 {'OFFENSE_DESCRIPTION': 'INVESTIGATE PROPERTY',
  'OCCURRED_ON_DATE': '2023-03-31 17:14:00+00'},
 {'OFFENSE_DESCRIPTION': 'ASSAULT - AGGRAVATED',
  'OCCURRED_ON_DATE': '2023-01-26 09:00:00+00'},
 {'OFFENSE_DESCRIPTION': 'INVESTIGATE PERSON',
  'OCCURRED_ON_DATE': '2023-01-29 14:55:00+00'},
 {'OFFENSE_DESCRIPTION': 'INVESTIGATE PERSON',
  'OCCURRED_ON_DATE': '2023-02-07 11:05:00+00'},
 {'OFFENSE_DESCRIPTION': 'DISTURBING THE PEACE/ DISORDERLY CONDUCT/ GATHERING CAUSING ANNOYANCE/ NOISY PAR',
  'OCCURRED_ON_DATE': '2023-02-09 12:02:00+00'},
 {'OFFENSE_DESCRIPTION': 'INVESTIGATE PERSON',
  'OCCURRED_ON_DATE': '2023-05-09 12:38:00+00'},
 {'OFFENSE_DESCRIPTION': 'THREATS TO DO BODILY HARM',
  'OCCURRED_ON_DATE'

In [46]:
import pandas as pd

df = pd.DataFrame(records)
df.head()

,OFFENSE_DESCRIPTION,OCCURRED_ON_DATE
0,INVESTIGATE PERSON,2023-01-27 22:44:00+00
1,VERBAL DISPUTE,2023-01-17 20:21:00+00
2,INVESTIGATE PERSON,2023-01-24 00:00:00+00
3,INVESTIGATE PROPERTY,2023-03-31 17:14:00+00
4,ASSAULT - AGGRAVATED,2023-01-26 09:00:00+00


In [47]:
top_crimes = (
    df['OFFENSE_DESCRIPTION']
    .value_counts()
    .head(5)
    .reset_index()
)

top_crimes.columns = ['OFFENSE_DESCRIPTION', 'COUNT']
top_crimes

,OFFENSE_DESCRIPTION,COUNT
0,INVESTIGATE PERSON,1158
1,SICK ASSIST,1019
2,M/V - LEAVING SCENE - PROPERTY DAMAGE,510
3,LARCENY SHOPLIFTING,443
4,ASSAULT - SIMPLE,426


In [48]:
import plotly.express as px

fig1 = px.bar(
    top_crimes,
    x='OFFENSE_DESCRIPTION',
    y='COUNT',
    title='Top 5 Crime Types in Boston',
)

fig1.show()

In [54]:
df['OCCURRED_ON_DATE'] = pd.to_datetime(df['OCCURRED_ON_DATE'], errors='coerce')
df = df.dropna(subset=['OCCURRED_ON_DATE', 'OFFENSE_DESCRIPTION'])
df['HOUR'] = df['OCCURRED_ON_DATE'].dt.hour



In [57]:
top3 = top_crimes['OFFENSE_DESCRIPTION'].head(3).tolist()

df_top3 = df[df['OFFENSE_DESCRIPTION'].isin(top3)]

hourly_patterns = (
    df_top3.groupby(['HOUR', 'OFFENSE_DESCRIPTION'])
    .size()
    .reset_index(name='COUNT')
)

hourly_patterns.head()

,HOUR,OFFENSE_DESCRIPTION,COUNT
0,0,INVESTIGATE PERSON,102
1,0,M/V - LEAVING SCENE - PROPERTY DAMAGE,35
2,0,SICK ASSIST,70
3,1,INVESTIGATE PERSON,22
4,1,M/V - LEAVING SCENE - PROPERTY DAMAGE,11


In [58]:
fig2 = px.line(
    hourly_patterns,
    x='HOUR',
    y='COUNT',
    color='OFFENSE_DESCRIPTION',
    markers=True,
    title='Hourly Crime Patterns for Top 3 Crimes'
)

fig2.show()